In [3]:
import pandas as pd
df = pd.read_csv("CRMLSListing_with_rates.csv")
print(df.shape)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_18427/669314064.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_with_rates.csv")


(566673, 87)


Step 1  Process Date Fields

In [4]:

date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]
print(df[date_cols].dtypes)
print(df[date_cols].head(3))

CloseDate                   str
PurchaseContractDate        str
ListingContractDate         str
ContractStatusChangeDate    str
dtype: object
  CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0       NaN           2024-05-07          2024-01-01               2024-05-07
1       NaN                  NaN          2024-01-24               2024-01-24
2       NaN                  NaN          2024-01-12               2024-01-12


In [5]:
# Convert the date column to datetime.
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print(df[date_cols].dtypes)
print(df[date_cols].head(5))

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object
  CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0       NaT           2024-05-07          2024-01-01               2024-05-07
1       NaT                  NaT          2024-01-24               2024-01-24
2       NaT                  NaT          2024-01-12               2024-01-12
3       NaT                  NaT          2024-01-20               2024-01-20
4       NaT                  NaT          2024-01-12               2024-01-12


In [6]:
print("CloseDate missing:", df["CloseDate"].isnull().sum())
print("PurchaseContractDate missing:", df["PurchaseContractDate"].isnull().sum())

CloseDate missing: 398467
PurchaseContractDate missing: 291528


NaT values represent missing datetime information. 
These occur when certain events, such as closing or contract signing, have not yet taken place or are not recorded. 


In [7]:
# create date consistency flags

# A. listing date is after close date
df["listing_after_close_flag"] = (
    df["ListingContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["ListingContractDate"] > df["CloseDate"])
)

# B. purchase date is after close date
df["purchase_after_close_flag"] = (
    df["PurchaseContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["PurchaseContractDate"] > df["CloseDate"])
)

# C. negative timeline:
# listing date is after purchase date
df["negative_timeline_flag"] = (
    df["ListingContractDate"].notna() &
    df["PurchaseContractDate"].notna() &
    (df["ListingContractDate"] > df["PurchaseContractDate"])
)

#  check flag counts
print("Flag counts:")
print("listing_after_close_flag:", df["listing_after_close_flag"].sum())
print("purchase_after_close_flag:", df["purchase_after_close_flag"].sum())
print("negative_timeline_flag:", df["negative_timeline_flag"].sum())
print()

Flag counts:
listing_after_close_flag: 74
purchase_after_close_flag: 266
negative_timeline_flag: 277



In [8]:
print("Rows with listing_after_close_flag = 1")
print(df.loc[df["listing_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with purchase_after_close_flag = 1")
print(df.loc[df["purchase_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with negative_timeline_flag = 1")
print(df.loc[df["negative_timeline_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

Rows with listing_after_close_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
1406           2024-01-31           2024-01-15 2024-01-30
4140           2024-01-26           2024-01-10 2024-01-25
16407          2024-01-02           2023-12-04 2024-01-01
41873          2024-03-21           2024-03-03 2024-03-20
73360          2024-04-08           2024-03-27 2024-03-29

Rows with purchase_after_close_flag = 1
     ListingContractDate PurchaseContractDate  CloseDate
309           2024-01-31           2024-03-04 2024-01-31
919           2024-01-31           2024-02-05 2024-01-31
974           2024-01-29           2024-02-04 2024-01-29
1271          2024-01-29           2024-02-01 2024-01-29
4136          2024-01-25           2024-01-26 2024-01-25

Rows with negative_timeline_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
1406           2024-01-31           2024-01-15 2024-01-30
1712           2024-01-18           2024-01-05 2024-01-30
2665           202

In [9]:
# High-Missing Columns (≥90%)
missing_pct = df.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct >= 90].index.tolist()
print("High-missing cols:", len(high_missing_cols))

# Non-analytical Column
extra_drop = [
    "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName",
    "BuyerAgentMlsId", "BuyerAgencyCompensationType",
    "BuyerOfficeAOR",
]
extra_drop = [col for col in extra_drop if col in df.columns]
print("Manual drop cols:", len(extra_drop))

before_cols = df.shape[1]
# Execute Deletion
df = df.drop(columns=high_missing_cols)
df = df.drop(columns=extra_drop)

after_cols = df.shape[1]

print("===== Check Deletion =====")
print("Before:", before_cols)
print("After:", after_cols)
print("Removed:", before_cols - after_cols)

High-missing cols: 13
Manual drop cols: 11
===== Check Deletion =====
Before: 90
After: 66
Removed: 24


In [10]:
# ---------- Duplicate Column Check ----------
# 1. Suffix duplicates (.1, .2, etc.)
suffix_dups = [col for col in df.columns if col.endswith((".1", ".2", ".3"))]

# 2. Duplicate column names
name_dups = df.columns[df.columns.duplicated()].tolist()

# 3. Duplicate content columns
content_dups = []
cols = df.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            content_dups.append((cols[i], cols[j]))

# ---------- Print Summary ----------
print("===== Duplicate Column Summary =====")
print(f"Suffix duplicates (.1 etc): {len(suffix_dups)}")
print(f"Duplicate column names: {len(name_dups)}")
print(f"Duplicate content pairs: {len(content_dups)}")
print("\n===== Duplicate Content Pairs =====")
for col1, col2 in content_dups:
    print(f"{col1}  <->  {col2}")


===== Duplicate Column Summary =====
Suffix duplicates (.1 etc): 11
Duplicate column names: 0
Duplicate content pairs: 9

===== Duplicate Content Pairs =====
ListingKey  <->  ListingKeyNumeric
Latitude  <->  Latitude.1
Longitude  <->  Longitude.1
UnparsedAddress  <->  UnparsedAddress.1
PropertyType  <->  PropertyType.1
LivingArea  <->  LivingArea.1
ListPrice  <->  ListPrice.1
DaysOnMarket  <->  DaysOnMarket.1
BuyerOfficeName  <->  BuyerOfficeName.1


In [11]:
print("Check equality:", df["ListingKey"].equals(df["ListingKeyNumeric"]))

print("Different rows:", (df["ListingKey"] != df["ListingKeyNumeric"]).sum())

print("\nSample data:")
print(df[["ListingKey", "ListingKeyNumeric"]].head(5))

Check equality: True
Different rows: 0

Sample data:
   ListingKey  ListingKeyNumeric
0  1074973329         1074973329
1  1074954552         1074954552
2  1074936537         1074936537
3  1074917818         1074917818
4  1074143166         1074143166


In [12]:
# ---------- Drop Duplicate Columns ----------

# 1. Drop suffix duplicates (.1, .2, etc.)
df = df.drop(columns=suffix_dups, errors="ignore")
print(f"Dropped {len(suffix_dups)} suffix duplicate columns")

# 2. Drop confirmed redundant column
df = df.drop(columns=["ListingKeyNumeric"], errors="ignore")
print("Dropped ListingKeyNumeric (duplicate of ListingKey)")

before_rows = df.shape[0]



# Remove duplicate rows 
# =========================
before = len(df)
df = df.drop_duplicates()
after = len(df)

print("\n[Listing]")
print(f"Before : {before:,} rows")
print(f"After  : {after:,} rows")
print(f"Removed: {before - after:,} duplicate rows")

Dropped 11 suffix duplicate columns
Dropped ListingKeyNumeric (duplicate of ListingKey)

[Listing]
Before : 566,673 rows
After  : 566,673 rows
Removed: 0 duplicate rows


In [13]:
#Check to ensure that the data type is correct
num_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger"
]

print(df[num_cols].dtypes)
for col in num_cols:
    print(f"\n--- {col} sample values ---")
    print(df[col].head(3))
for col in num_cols:
    print(f"{col} missing after conversion:", df[col].isna().sum())

ClosePrice               float64
ListPrice                float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

--- ClosePrice sample values ---
0   NaN
1   NaN
2   NaN
Name: ClosePrice, dtype: float64

--- ListPrice sample values ---
0    1340000.0
1    2500000.0
2    3150000.0
Name: ListPrice, dtype: float64

--- LivingArea sample values ---
0    1301.0
1    2788.0
2    3250.0
Name: LivingArea, dtype: float64

--- DaysOnMarket sample values ---
0    127
1      1
2      1
Name: DaysOnMarket, dtype: int64

--- BedroomsTotal sample values ---
0    2.0
1    4.0
2    6.0
Name: BedroomsTotal, dtype: float64

--- BathroomsTotalInteger sample values ---
0    2.0
1    3.0
2    4.0
Name: BathroomsTotalInteger, dtype: float64
ClosePrice missing after conversion: 419976
ListPrice missing after conversion: 0
LivingArea missing after conversion: 580
DaysOnMarket missing after conversion: 0
Bedrooms

In [14]:
# numeric columns
numeric_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "ListPrice",
    "LivingArea",
    "LotSizeAcres",
    "DaysOnMarket",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "Latitude",
    "Longitude",
    "AssociationFee",
    "ParkingTotal",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "rate_30yr_fixed"
]
# convert to numeric
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
# check data types
print("\nNumeric Column Types:")
print(df[numeric_cols].dtypes)

# preview
print("\nNumeric Columns Preview:")
print(df[numeric_cols].head(3))


Numeric Column Types:
ClosePrice               float64
OriginalListPrice        float64
ListPrice                float64
LivingArea               float64
LotSizeAcres             float64
DaysOnMarket               int64
YearBuilt                float64
BathroomsTotalInteger    float64
BedroomsTotal            float64
Latitude                 float64
Longitude                float64
AssociationFee           float64
ParkingTotal             float64
GarageSpaces             float64
LotSizeSquareFeet        float64
rate_30yr_fixed          float64
dtype: object

Numeric Columns Preview:
   ClosePrice  OriginalListPrice  ListPrice  LivingArea  LotSizeAcres  \
0         NaN          1340000.0  1340000.0      1301.0        4.0831   
1         NaN          2500000.0  2500000.0      2788.0        0.1217   
2         NaN          3150000.0  3150000.0      3250.0        0.2159   

   DaysOnMarket  YearBuilt  BathroomsTotalInteger  BedroomsTotal   Latitude  \
0           127     1964.0           

Step 3: Outlier Check (Invalid Values)

In [15]:
# Invalid value flags 
# -------------------------------

df["invalid_close_price_flag"] = (df["ClosePrice"] <= 0)
df["invalid_living_area_flag"] = (df["LivingArea"] <= 0)
df["invalid_days_on_market_flag"] = (df["DaysOnMarket"] < 0)
df["invalid_bedrooms_flag"] = (df["BedroomsTotal"] < 0)
df["invalid_bathrooms_flag"] = (df["BathroomsTotalInteger"] < 0)

# count
print("Invalid Value Counts:")
print(df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag"
    ]
].sum())

Invalid Value Counts:
invalid_close_price_flag         0
invalid_living_area_flag       376
invalid_days_on_market_flag     29
invalid_bedrooms_flag            0
invalid_bathrooms_flag           0
dtype: int64


Step 4：Geographic Data Checks

In [16]:
# -------------------------------
# Geographic flags
# -------------------------------

# 1️ Missing Values
df["missing_geo_flag"] = (
    df["Latitude"].isna() | df["Longitude"].isna()
)

# 2️.Zero Values ​​(Dummy Data)
df["zero_geo_flag"] = (
    (df["Latitude"] == 0) | (df["Longitude"] == 0)
)

# 3.Incorrect longitude direction (California should be negative).
df["invalid_longitude_flag"] = (
    df["Longitude"] > 0
)

# 4️. Outside the California area (approximate range)
df["out_of_ca_flag"] = (
    (df["Latitude"] < 32) | (df["Latitude"] > 42) |
    (df["Longitude"] < -125) | (df["Longitude"] > -114)
)

# Count issues
print("Geographic Issues Count:")
print("missing_geo_flag:", df["missing_geo_flag"].sum())
print("zero_geo_flag:", df["zero_geo_flag"].sum())
print("invalid_longitude_flag:", df["invalid_longitude_flag"].sum())
print("out_of_ca_flag:", df["out_of_ca_flag"].sum())

Geographic Issues Count:
missing_geo_flag: 80441
zero_geo_flag: 65
invalid_longitude_flag: 86
out_of_ca_flag: 301


In [29]:
# -------------------------------
# Clean data: numeric + date + geo
# -------------------------------

before_rows = len(df)

df = df[
    # numeric validity
    (df["ClosePrice"] > 0) &
    (df["LivingArea"] > 0) &
    (df["DaysOnMarket"] >= 0) &
    (
        (df["BedroomsTotal"].isna()) |
        (df["BedroomsTotal"] >= 0)
    ) &
    (
        (df["BathroomsTotalInteger"].isna()) |
        (df["BathroomsTotalInteger"] >= 0)
    ) &

    # date validity
    (df["listing_after_close_flag"] == 0) &
    (df["purchase_after_close_flag"] == 0) &
    (df["negative_timeline_flag"] == 0) &

    # geo validity
    (df["zero_geo_flag"] == 0) &
    (df["invalid_longitude_flag"] == 0) &
    (df["out_of_ca_flag"] == 0)
].copy()

after_rows = len(df)

print("\nFinal Cleaning Summary:")
print("Rows before cleaning:", before_rows)
print("Rows after cleaning :", after_rows)
print("Rows removed        :", before_rows - after_rows)


Final Cleaning Summary:
Rows before cleaning: 146049
Rows after cleaning : 146049
Rows removed        : 0


In [ ]:
flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag"
]

df[flag_cols].sum().sort_values(ascending=False)

missing_geo_flag               15671
listing_after_close_flag           0
purchase_after_close_flag          0
negative_timeline_flag             0
invalid_close_price_flag           0
invalid_living_area_flag           0
invalid_days_on_market_flag        0
invalid_bedrooms_flag              0
invalid_bathrooms_flag             0
zero_geo_flag                      0
invalid_longitude_flag             0
out_of_ca_flag                     0
dtype: int64

week-6

In [36]:
import pandas as pd
import numpy as np

# Key Metrics to Create
# =============================

# 1. Price Ratio
df["price_ratio"] = np.where(
    df["OriginalListPrice"] > 0,
    df["ClosePrice"] / df["OriginalListPrice"],
    np.nan
)

# 2. Close-to-Original-List Ratio
df["close_to_original_list_ratio"] = np.where(
    df["OriginalListPrice"] > 0,
    df["ClosePrice"] / df["OriginalListPrice"],
    np.nan
)

# 3. Price Per Square Foot
df["price_per_sqft"] = np.where(
    df["LivingArea"] > 0,
    df["ClosePrice"] / df["LivingArea"],
    np.nan
)

# 4. Year / Month / YrMo
df["close_year"] = df["CloseDate"].dt.year
df["close_month"] = df["CloseDate"].dt.month
df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

# 5. Listing-to-Contract Days
df["listing_to_contract_days"] = (
    df["PurchaseContractDate"] - df["ListingContractDate"]
).dt.days

# 6. Contract-to-Close Days
df["contract_to_close_days"] = (
    df["CloseDate"] - df["PurchaseContractDate"]
).dt.days

# Sample output table
# =============================

sample_cols = [
    "CloseDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "DaysOnMarket",
    "YrMo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

print("\nSample engineered metrics:")
print(df[sample_cols].head(5))




# Save outputs
# =============================

#df.to_csv("CRMLSListing_Week6_Features.csv", index=False)
print("Saved: CRMLSListing_Week6_Features.csv")


Sample engineered metrics:
    CloseDate  ClosePrice  OriginalListPrice  LivingArea  price_ratio  \
9  2024-05-01   1262555.0          1249888.0      1899.0     1.010135   
14 2024-03-15    650000.0           650000.0      1400.0     1.000000   
18 2024-05-02    962500.0          1025000.0      1130.0     0.939024   
20 2024-04-30    725000.0           725000.0      1250.0     1.000000   
33 2024-04-09    730000.0           730000.0      1266.0     1.000000   

    close_to_original_list_ratio  price_per_sqft  DaysOnMarket     YrMo  \
9                       1.010135      664.852554             0  2024-05   
14                      1.000000      464.285714             0  2024-03   
18                      0.939024      851.769912             0  2024-05   
20                      1.000000      580.000000             0  2024-04   
33                      1.000000      576.619273            35  2024-04   

    listing_to_contract_days  contract_to_close_days  
9                       20.

In [44]:
key_metrics = [
    'price_ratio',
    'price_per_sqft',
    'DaysOnMarket',
    'close_to_original_list_ratio',
    'listing_to_contract_days',
    'contract_to_close_days'
]

for m in key_metrics:
    print(f"{m}: {'exists' if m in df.columns else ' not found'}")

price_ratio: exists
price_per_sqft: exists
DaysOnMarket: exists
close_to_original_list_ratio: exists
listing_to_contract_days: exists
contract_to_close_days: exists


In [38]:
# Segment summary table
# 1.Grouped by CountyOrParish
# =============================

county_summary = (
    df.groupby("CountyOrParish")
    .agg(
        total_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("DaysOnMarket", "mean"),
        median_price_ratio=("price_ratio", "median"),
        average_listing_to_contract_days=("listing_to_contract_days", "mean"),
        average_contract_to_close_days=("contract_to_close_days", "mean")
    )
    .reset_index()
    .sort_values(by="median_close_price", ascending=False)
)

# Save outputs
# =============================
#county_summary.to_csv("Listing_county_segment_summary.csv", index=False)
print("Saved: Listing_county_segment_summary.csv")
pd.read_csv("Listing_county_segment_summary.csv")

Saved: Listing_county_segment_summary.csv


,CountyOrParish,total_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_price_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,San Mateo,3137,1758888.0,2.253103e+06,1094.470046,15.573797,1.032609,14.783870,19.490915
1,Santa Clara,7640,1644125.0,1.973980e+06,1002.880891,14.235471,1.042176,13.449215,20.748168
2,Marin,58,1298500.0,1.538981e+06,732.285750,21.931034,1.000000,22.982759,23.603448
3,San Francisco,396,1250000.0,1.402380e+06,938.883948,20.323232,1.029660,20.681818,20.202020
4,Santa Cruz,1146,1225000.0,1.358880e+06,756.674137,24.018325,1.000000,22.416230,21.675393
5,Alameda,8514,1187500.0,1.314600e+06,719.092525,17.940686,1.033333,18.167959,21.866690
6,Orange,16634,1175000.0,1.532075e+06,679.341890,20.770109,1.000000,27.219069,26.536492
7,Los Angeles,34921,915000.0,1.295495e+06,620.891161,23.804330,1.000000,27.933587,28.714810
8,Monterey,1443,910000.0,1.379744e+06,600.000000,23.128205,1.000000,22.962578,24.928621
9,San Diego,20127,900000.0,1.317319e+06,597.426471,20.004223,1.000000,24.399334,25.251515


In [ ]:
# =============================
# PropertyType + PropertySubType
# =============================

property_summary = (
    df.groupby(
        ["PropertyType", "PropertySubType"]
    )
    .agg(
        total_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("DaysOnMarket", "mean"),
        median_price_ratio=("price_ratio", "median")
    )
    .reset_index()
    .sort_values(
        by="median_close_price",
        ascending=False
    )
)

property_summary.to_csv(
   "Listing_property_segment_summary.csv",
    index=False
)
print("Saved: Listing_property_segment_summary.csv")
pd.read_csv("Listing_property_segment_summary.csv")

Saved: Listing_property_segment_summary.csv


,PropertyType,PropertySubType,total_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_price_ratio
0,Residential,Farm,4,2031168.000,2.240584e+06,903.091337,52.750000,0.909713
1,Residential,Quadruplex,44,1287500.000,1.420486e+06,416.149134,34.636364,0.975417
2,Residential,Triplex,100,1065000.000,1.315369e+06,462.583197,29.180000,0.985638
3,Residential,SingleFamilyResidence,110084,930000.000,1.297215e+06,566.304034,22.055321,1.000000
4,Residential,Duplex,713,927500.000,1.253217e+06,568.942436,22.354839,1.000000
5,Residential,CoOwnership,4,844000.000,1.829500e+06,652.369138,19.250000,1.005562
6,Residential,Townhouse,8933,810000.000,9.965075e+05,567.466667,22.938542,1.000000
7,Residential,DeededParking,2,795000.000,7.950000e+05,500.190659,35.500000,0.988885
8,Residential,Timeshare,9,790000.000,6.624444e+05,312.871287,38.333333,1.000000
9,Residential,Loft,4,655444.445,6.037222e+05,494.627182,21.500000,1.000705


In [ ]:
# =============================
# CountyOrParish + MLSAreaMajor
# =============================

county_area_summary = (
    df.groupby(
        ["CountyOrParish", "MLSAreaMajor"]
    )
    .agg(
        total_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("DaysOnMarket", "mean"),
        median_price_ratio=("price_ratio", "median")
    )
    .reset_index()
    .sort_values(
        by="median_close_price",
        ascending=False
    )
)

county_area_summary.to_csv(
    "Listing_county_area_segment_summary.csv",
    index=False
)
print("Saved: Listing_county_area_segment_summary.csv")
pd.read_csv("Listing_county_area_segment_summary.csv")

Saved: Listing_county_area_segment_summary.csv


,CountyOrParish,MLSAreaMajor,total_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_price_ratio
0,Los Angeles,C32 - Malibu Beach,21,11650000.0,1.427898e+07,2734.445042,48.952381,0.917431
1,Orange,SH - Shady Canyon,7,11000000.0,1.131711e+07,1455.604076,14.571429,1.000000
2,Orange,CR - Crystal Cove,10,10867500.0,1.569360e+07,2672.113498,18.700000,1.000000
3,Santa Barbara,MCTO - Montecito,38,7000000.0,7.638916e+06,1943.904471,30.578947,0.974302
4,Los Angeles,144 - Manhattan Bch Hill,10,6116500.0,6.653300e+06,1910.105021,15.500000,1.005263
...,...,...,...,...,...,...,...,...
1107,Kern,BFSH - Bodfish,3,105000.0,8.466667e+04,78.125000,17.666667,0.933333
1108,Kern,INYO - Inyokern,1,97000.0,9.700000e+04,67.361111,4.000000,1.000000
1109,San Bernardino,TRNA - Trona,7,75000.0,6.442857e+04,65.217391,27.142857,1.000000
1110,San Bernardino,BAKR - Baker,1,56000.0,5.600000e+04,41.666667,69.000000,0.949153


In [42]:
# =============================
# ListOfficeName + BuyerOfficeName
# =============================

office_summary = (
    df.groupby(
        ["ListOfficeName", "BuyerOfficeName"]
    )
    .agg(
        total_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("DaysOnMarket", "mean"),
        median_price_ratio=("price_ratio", "median")
    )
    .reset_index()
    .sort_values(
        by="total_sales",
        ascending=False
    )
)

office_summary.to_csv(
    "Listing_office_segment_summary.csv",
    index=False
)
print("Saved: Listing_office_segment_summary.csv")
pd.read_csv("Listing_office_segment_summary.csv")

Saved: Listing_office_segment_summary.csv


,ListOfficeName,BuyerOfficeName,total_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_price_ratio
0,Compass,Compass,3260,1599500.0,2.099091e+06,852.094925,16.531288,1.000000
1,Coldwell Banker Realty,Coldwell Banker Realty,1451,1295000.0,1.766256e+06,742.753623,17.178498,1.000000
2,Coldwell Banker Realty,Compass,637,1555000.0,1.933316e+06,850.000000,21.591837,1.000667
3,Compass,Coldwell Banker Realty,570,1650000.0,2.175208e+06,882.936137,20.228070,1.000000
4,Keller Williams Realty,Keller Williams Realty,536,930000.0,1.128303e+06,559.468233,17.162313,1.000000
...,...,...,...,...,...,...,...,...
84184,GE Realty,Coldwell Banker Realty,1,1390000.0,1.390000e+06,636.155606,28.000000,0.993567
84185,GE Realty,Aviara Real Estate,1,1200000.0,1.200000e+06,557.103064,45.000000,0.960001
84186,GE Dean and Associates,"eXp Realty of Greater Los Angeles, Inc.",1,1300000.0,1.300000e+06,606.909430,33.000000,1.000000
84187,GE Dean and Associates,"Zutila, Inc",1,920000.0,9.200000e+05,622.462788,8.000000,1.023359
